In [32]:
import pandas as pd
import datetime as dt
import ast

pl1_file = pd.read_csv('Data/real-data/DYAD_10_SCAN_101_abstract-images.csv')
pl2_file = pd.read_csv('Data/real-data/DYAD_10_SCAN_102_abstract-images.csv')
dyad_id = 10
pl1_director = pl1_file[pl1_file['Role'] == 'director']
pl1_guessor = pl1_file[pl1_file['Role'] == 'guessor']
pl2_director = pl2_file[pl2_file['Role'] == 'director']
pl2_guessor = pl2_file[pl2_file['Role'] == 'guessor']

In [ ]:
pl1_rounds = []
pl2_rounds = []
for i in range(len(pl2_director.index)):
    round = {'director': list(pl2_director.iloc[i][9:15]), 
             'guessor': list(pl1_guessor.iloc[i][9:15]), 
             'inputs': list(pl1_guessor.iloc[i][15:27:2]),
             'rts': list(pl1_guessor.iloc[i][16:28:2]),
             'start_time': pl2_director.iloc[i]['Round_start_time'],
             'end_time': pl2_director.iloc[i]['Round_end_time'],
             'control': pl1_guessor.iloc[i]['Control?'],
             'image_time': list(pl2_director.iloc[i][16:28:2])}
    pl1_rounds.append(round)
for i in range(len(pl1_director.index)):
    round = {'director': list(pl1_director.iloc[i][9:15]), 
             'guessor': list(pl2_guessor.iloc[i][9:15]), 
             'inputs': list(pl2_guessor.iloc[i][15:27:2]),
             'rts': list(pl2_guessor.iloc[i][16:28:2]),
             'start_time': pl1_director.iloc[i]['Round_start_time'],
             'end_time': pl1_director.iloc[i]['Round_end_time'],
             'control': pl2_guessor.iloc[i]['Control?'],
             'image_time': list(pl1_director.iloc[i][16:28:2])}
    pl2_rounds.append(round)

In [55]:
def calc_rt(start, irt, diff, i, img) :    
    rt = dt.datetime.strptime(irt, '%H:%M:%S')
    if dyad_id < 10 :
        rt_act = rt - (start + dt.timedelta(0,(diff.total_seconds()/6)*i))
    else :
        img_t = dt.datetime.strptime(img, '%H:%M:%S')
        rt_act = rt - img_t
        print(img_t, rt, rt_act)
    return rt_act.total_seconds()

def check_correct(director, guessor, inputs, rts, start, diff, image_times) :
    #see when guessor image occurs in director list, then compare to answer in box
    graded_set = []
    #get accuracy % and round rt
    round_accuracy = 0
    round_rt = 0
    for i, d in enumerate(director) :
        accurate = False
        correct_indices = []
        correct_rt = []
        all_incorrect_responses = []
        all_incorrect_rt = []
        num_responses = 0
        image_time = image_times[i]
        for j, g, in enumerate(guessor) :
            incorrect_indices = []
            incorrect_responses = []
            incorrect_rt = []
            inp = ast.literal_eval(inputs[j])
            if str(i+1) in inp :
                if d == g :    
                    correct_indices = [k for k, x in enumerate(inp) if x == str(i+1)]    
                    incorrect_responses = [x for x in inp if x != str(i+1)]
                    num_responses = len(inp)       
                    crt_set = [ast.literal_eval(rts[j])[k] for k in correct_indices]
                    for rt in crt_set :
                        correct_rt.append(calc_rt(start, rt, diff, i, image_time))                                     
                    if correct_indices[-1] == len(inp)-1 :
                        accurate = True
                else :
                    incorrect_indices = [k for k, x in enumerate(inp) if x == str(i+1)]    
                    irt_set = [ast.literal_eval(rts[j])[k] for k in incorrect_indices]
                    image_time_set = [image_times[k] for k in incorrect_indices]
                    for ind, rt, img_t in zip(incorrect_indices, irt_set, image_time_set) :
                        incorrect_rt.append(calc_rt(start, rt, diff, ind, img_t))
            else :
                if d == g :
                    num_responses = len(inp)
                    incorrect_responses = inp
            [all_incorrect_rt.append(x) for x in incorrect_rt]    
            [all_incorrect_responses.append(x) for x in incorrect_responses]        
        if accurate :
            round_accuracy += 1
            if correct_rt[::-1][0] <= 20 :
                round_rt += correct_rt[::-1][0]
            else :
                round_rt += 20
        else :
            round_rt += 20
        
        graded = {'image': d, 'correct?': accurate, 'correct rt': correct_rt, 'incorrect rt': all_incorrect_rt,
                  'incorrect resp box': all_incorrect_responses, 'number resp box': num_responses}
        print('graded: ', graded)
        graded_set.append(graded)
    print('ROUND ', round_rt, round_accuracy/6)        
    return round_accuracy/6, round_rt, graded_set       

In [56]:
pl1_answers = []
pl1_accs = []
pl1_rts = []
pl2_answers = []
pl2_accs = []
pl2_rts = []
for i in range(len(pl1_rounds)) :
    pl1_st = dt.datetime.strptime(pl1_rounds[i]['start_time'], '%H:%M:%S')
    pl1_end = dt.datetime.strptime(pl1_rounds[i]['end_time'], '%H:%M:%S')
    pl1_diff = pl1_end - pl1_st
    if pl1_rounds[i]['control'] == False :
        pl1_acc, pl1_rt, pl1_ans = check_correct(pl1_rounds[i]['director'], pl1_rounds[i]['guessor'], pl1_rounds[i]['inputs'], 
                                pl1_rounds[i]['rts'], pl1_st, pl1_diff, pl1_rounds[i]['image_time'])
        pl1_answers.append(pl1_ans)
        pl1_accs.append(pl1_acc)
        pl1_rts.append(pl1_rt)
    else :
        pl1_answers.append(['control'])
        pl1_accs.append('N/A')
        pl1_rts.append('N/A')

    pl2_st = dt.datetime.strptime(pl2_rounds[i]['start_time'], '%H:%M:%S')
    pl2_end = dt.datetime.strptime(pl2_rounds[i]['end_time'], '%H:%M:%S')
    pl2_diff = pl2_end - pl2_st 
    if pl2_rounds[i]['control'] == False :
        pl2_acc, pl2_rt, pl2_ans = check_correct(pl2_rounds[i]['director'], pl2_rounds[i]['guessor'], pl2_rounds[i]['inputs'], 
                                pl2_rounds[i]['rts'], pl2_st, pl2_diff, pl2_rounds[i]['image_time'])
        pl2_answers.append(pl2_ans)
        pl2_accs.append(pl2_acc)
        pl2_rts.append(pl2_rt)
    else :
        pl2_answers.append(['control'])
        pl2_accs.append('N/A')
        pl2_rts.append('N/A')

pl1_answer_file = pd.concat([pl1_guessor[pl1_guessor.columns[:5]].reset_index(drop=True), pd.Series(pl1_accs, name='Accuracy'), 
                             pd.Series(pl1_rts, name='RT'), pd.DataFrame(pl1_answers)], axis=1).drop(columns=['Role', 'Control?'])
pl1_answer_file.to_csv('Data/answer-files/DYAD_10_SCAN_101_answers.csv', index=False)
pl2_answer_file = pd.concat([pl2_guessor[pl2_guessor.columns[:5]].reset_index(drop=True), pd.Series(pl2_accs, name='Accuracy'), 
                             pd.Series(pl2_rts, name='RT'), pd.DataFrame(pl2_answers)], axis=1).drop(columns=['Role', 'Control?'])
pl2_answer_file.to_csv('Data/answer-files/DYAD_10_SCAN_102_answers.csv', index=False)

1900-01-01 14:15:21 1900-01-01 14:15:52 0:00:31
1900-01-01 14:15:21 1900-01-01 14:17:19 0:01:58
1900-01-01 14:15:21 1900-01-01 14:15:57 0:00:36
graded:  {'image': '10.png', 'correct?': True, 'correct rt': [118.0], 'incorrect rt': [31.0, 36.0], 'incorrect resp box': ['6'], 'number resp box': 2}
1900-01-01 14:15:41 1900-01-01 14:15:49 0:00:08
graded:  {'image': '21.png', 'correct?': True, 'correct rt': [8.0], 'incorrect rt': [], 'incorrect resp box': [], 'number resp box': 1}
1900-01-01 14:16:02 1900-01-01 14:16:18 0:00:16
1900-01-01 14:15:21 1900-01-01 14:16:13 0:00:52
graded:  {'image': '11.png', 'correct?': True, 'correct rt': [16.0], 'incorrect rt': [52.0], 'incorrect resp box': [], 'number resp box': 1}
1900-01-01 14:16:22 1900-01-01 14:16:40 0:00:18
graded:  {'image': '54.png', 'correct?': True, 'correct rt': [18.0], 'incorrect rt': [], 'incorrect resp box': ['3'], 'number resp box': 2}
1900-01-01 14:16:42 1900-01-01 14:16:50 0:00:08
graded:  {'image': '42.png', 'correct?': True, '